In [ ]:
#all imports needed for the notebook
from config import DATA_DIR
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import glob, os
from matplotlib.animation import FuncAnimation
from matplotlib import rcParams
from pathlib import Path
from IPython.display import HTML
import plotly.graph_objects as go

In [ ]:
#checking the content of one of the npz files


path = Path(DATA_DIR) / "masked_scans0" / "1.3.6.1.4.1.14519.5.2.1.6279.6001.134996872583497382954024478441.npz"
data = np.load(path)
print(data.files)
vol = data["volume"]
print(vol.shape, vol.dtype, vol.min(), vol.max())
print(data['origin'], data['spacing'], data['shape'])

In [ ]:
#checking how many unique seriesuids with positive nodules there are in the candidates_index.csv file
#mostly used to find an interesting ct scan to visualize


index = pd.read_csv(Path(DATA_DIR) / "masked_scans1" / "candidates_index.csv")
pos_all = index[index["class"] == 1]
seriesuid = pos_all["seriesuid"].iloc
# how many entries in seriesuid are duplicates?
unique_seriesuids = set(seriesuid)
print(f"Number of unique seriesuids with positive nodules: {len(unique_seriesuids)}")
seriesuid_list = list(seriesuid)
print(f"Number of entries in seriesuid: {len(seriesuid_list)}")
#which seriesuid appears most frequently?
from collections import Counter
counter = Counter(seriesuid_list)
most_common = counter.most_common(1)
print(f"Most common seriesuid: {most_common[0][0]} with {most_common[0][1]} entries")

In [ ]:
#visualizes all positive nodules in the most common seriesuid as red circles on top of the ct scan slice where they are located

path = Path(DATA_DIR) / "masked_scans1" / f"{most_common[0][0]}.npz"
data = np.load(path)
vol = data["volume"]
seriesuid = path.stem
idx = pd.read_csv(path.parent / "candidates_index.csv")
pos = idx[(idx["seriesuid"] == seriesuid) & (idx["class"] == 1)]

for _, r in pos.iterrows():
    z, y, x = int(r["voxel_z"]), int(r["voxel_y"]), int(r["voxel_x"])
    plt.imshow(vol[z], cmap="gray", vmin=-1000, vmax=400)
    plt.colorbar()
    plt.scatter([x], [y], s=200, facecolors="none", edgecolors="red")
    plt.title(f"positive nodule z={z}")
    plt.show()

In [ ]:
#visualizes the ct scan with the most positive nodules as an animation, where the red circles fade in and out 
#depending on how close the current slice is to the nodule location

path = Path(DATA_DIR) / "masked_scans1" / f"{most_common[0][0]}.npz"
data = np.load(path)
vol = data["volume"]
seriesuid = path.stem
idx = pd.read_csv(path.parent / "candidates_index.csv")
pos = idx[(idx["seriesuid"] == seriesuid) & (idx["class"] == 1)]

nodules = [(int(r["voxel_z"]), int(r["voxel_y"]), int(r["voxel_x"])) for _, r in pos.iterrows()]

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(vol[0], cmap="gray", vmin=-1000, vmax=400)

def update(z):
    im.set_data(vol[z])
    # remove previous markers/labels, keep the image
    for artist in ax.collections + ax.texts:
        artist.remove()
    for nz, ny, nx in nodules:
        dist = abs(nz - z)
        alpha = max(0.15, 1.0 - dist / 15)
        ax.scatter([nx], [ny], s=200, facecolors="none", edgecolors="red", alpha=alpha)
        ax.text(nx + 12, ny, f"z={nz}", color="red", fontsize=8, alpha=alpha, va="center")
    ax.set_title(f"slice z={z}")
    return [im]

anim = FuncAnimation(fig, update, frames=vol.shape[0], interval=120)
plt.close()
rcParams["animation.embed_limit"] = 100
HTML(anim.to_jshtml())

In [ ]:
#visualizes the positive nodules in the most common seriesuid as red points in a 3d scatter plot

path = Path(DATA_DIR) / "masked_scans1" / f"{most_common[0][0]}.npz"
data = np.load(path)
vol = data["volume"]
seriesuid = path.stem
idx = pd.read_csv(path.parent / "candidates_index.csv")
pos = idx[(idx["seriesuid"] == seriesuid) & (idx["class"] == 1)]

fig = go.Figure(go.Scatter3d(
    x=pos["voxel_x"], y=pos["voxel_y"], z=pos["voxel_z"],
    mode="markers",
    marker=dict(size=5, color="red"),
    text=[f"z={int(z)}" for z in pos["voxel_z"]],
))
fig.update_layout(scene=dict(aspectmode="data"))
fig.show()

In [ ]:
#ATTENTION: this code takes ages to run and sometimes crashes the notebook but looks nice
# visualizes the ct scan as an isosurface

small = vol[::2, ::2, ::2]
zz, yy, xx = np.mgrid[0:small.shape[0], 0:small.shape[1], 0:small.shape[2]]

fig = go.Figure(go.Isosurface(
    x=xx.flatten(), y=yy.flatten(), z=zz.flatten(),
    value=small.flatten(),
    isomin=-300, isomax=300,
    surface_count=2,
    opacity=0.3,
    caps=dict(x_show=False, y_show=False, z_show=False),
))
fig.show()